Работа с табличными данными

In [1]:
import pandas as pd
import numpy as np
import copy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from dotenv import load_dotenv
import wandb
import os
import logging
import time
import torch.nn as nn
import random
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import *
import pickle

In [2]:
df = pd.read_csv('dataset/train.csv', sep="|")
df

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
0,5,1054,54.70,7,0,3,0.027514,0.051898,0.241379,0
1,3,108,27.36,5,2,4,0.129630,0.253333,0.357143,0
2,3,1516,62.16,3,10,5,0.008575,0.041003,0.230769,0
3,6,1791,92.31,8,4,4,0.016192,0.051541,0.275862,0
4,5,430,81.53,3,7,2,0.062791,0.189605,0.111111,0
...,...,...,...,...,...,...,...,...,...,...
1874,1,321,76.03,8,7,2,0.071651,0.236854,0.347826,0
1875,1,397,41.89,5,5,0,0.065491,0.105516,0.192308,1
1876,4,316,41.83,5,8,1,0.094937,0.132373,0.166667,0
1877,2,685,62.68,1,6,2,0.035036,0.091504,0.041667,0


In [3]:
df.dtypes

trustLevel                     int64
totalScanTimeInSeconds         int64
grandTotal                   float64
lineItemVoids                  int64
scansWithoutRegistration       int64
quantityModifications          int64
scannedLineItemsPerSecond    float64
valuePerSecond               float64
lineItemVoidsPerPosition     float64
fraud                          int64
dtype: object

In [4]:
df.isna().sum()

trustLevel                   0
totalScanTimeInSeconds       0
grandTotal                   0
lineItemVoids                0
scansWithoutRegistration     0
quantityModifications        0
scannedLineItemsPerSecond    0
valuePerSecond               0
lineItemVoidsPerPosition     0
fraud                        0
dtype: int64

In [5]:
df.fraud.value_counts()

fraud
0    1775
1     104
Name: count, dtype: int64

In [6]:
y = df["fraud"]
X = df.drop(columns=["fraud"])

In [7]:
df.describe()

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
count,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000
mean,3.401809,932.153273,50.864492,5.469931,4.904204,2.525279,0.058138,0.201746,0.745404,0.055349
std,1.709404,530.144640,28.940202,3.451169,3.139697,1.695472,0.278512,1.242135,1.327241,0.228720
min,1.000000,2.000000,0.010000,0.000000,0.000000,0.000000,0.000548,0.000007,0.000000,0.000000
25%,2.000000,474.500000,25.965000,2.000000,2.000000,1.000000,0.008384,0.027787,0.160000,0.000000
50%,3.000000,932.000000,51.210000,5.000000,5.000000,3.000000,0.016317,0.054498,0.350000,0.000000
75%,5.000000,1397.000000,77.285000,8.000000,8.000000,4.000000,0.032594,0.107313,0.666667,0.000000
max,6.000000,1831.000000,99.960000,11.000000,10.000000,5.000000,6.666667,37.870000,11.000000,1.000000


## Предобработка данных

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [9]:
X_train.shape

(1503, 9)

In [10]:
X_val.shape

(376, 9)

In [11]:
X_test = pd.read_csv("dataset/test.csv", sep="|")
X_test

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition
0,4,467,88.48,4,8,4,0.014989,0.189465,0.571429
1,3,1004,58.99,7,6,1,0.026892,0.058755,0.259259
2,1,162,14.00,4,5,4,0.006173,0.086420,4.000000
3,5,532,84.79,9,3,4,0.026316,0.159380,0.642857
4,5,890,42.16,4,0,0,0.021348,0.047371,0.210526
...,...,...,...,...,...,...,...,...,...
498116,4,783,59.10,2,2,0,0.012771,0.075479,0.200000
498117,1,278,98.90,9,5,4,0.050360,0.355755,0.642857
498118,3,300,5.41,6,6,4,0.030000,0.018033,0.666667
498119,2,1524,33.97,2,5,3,0.005906,0.022290,0.222222


In [12]:
y_test = pd.read_csv("dataset/DMC-2019-realclass.csv", sep="|")["fraud"]
y_test.value_counts()

fraud
0    474394
1     23727
Name: count, dtype: int64

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [14]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

In [15]:
y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [16]:
BATCH_SIZE = 64

In [17]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=1024)

вес для редкого класса, чтобы модель его не пропускала

In [18]:
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

In [19]:
pos_weight = n_neg / n_pos
pos_weight

tensor(17.1084)

## Подготовка

In [20]:
load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "gp5")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")


In [21]:
wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/maxsafonov/.netrc
wandb: Currently logged in as: gigantina-ru (gigantina-ru-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [22]:
def stop_logging():
    logger = logging.getLogger()
    for handler in logger.handlers:
        handler.flush()
        handler.close()
        logger.removeHandler(handler)

def new_log_file():
    stop_logging()
    timestamp = str(time.time()).replace('.', '_')
    log_file = f'part_2_{timestamp}.log'
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True
    )
    logging.info("Начал логгировать новый запуск")
    return log_file

In [23]:
def evaluate_model(model, X, y, threshold=0.5, name="dataset"):
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED) 

    model.eval()

    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)

    y_true = y.numpy().ravel()
    y_prob = probs.numpy().ravel()
    y_pred = (y_prob > threshold).astype(int)

    roc_auc = roc_auc_score(y_true, y_prob)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f05 = fbeta_score(y_true, y_pred, beta=0.5)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    profit = tp * 5 - fp * 25 - fn * 5

    print(name)
    print("ROC-AUC:", round(roc_auc, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F0.5-score:", round(f05, 4))
    print("Profit:", profit)
    print()

    return {
        "dataset": name,
        "threshold": threshold,
        "roc_auc": roc_auc,
        "precision": precision,
        "recall": recall,
        "f05": f05,
        "profit": profit
    }

In [24]:
def train_model(model, train_loader, X_valid, y_valid, loss_fn, optimizer, 
    epochs=30, threshold=0.5,model_name="model"):
    
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED) 

    best_metric = -10**9
    best_epoch = 0
    best_state = None

    history = []
    logging.info(f"Начали обучение {model_name}")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0

        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)

        valid_metrics = evaluate_model(model, X_valid, y_valid, threshold=threshold,
            name=f"{model_name} | valid epoch {epoch}")
            
        current_metric = valid_metrics["roc_auc"]
        
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        print(f"Эпоха {epoch} | " f"Train Loss: {epoch_loss:.4f} | " 
        f"Val ROC-AUC: {valid_metrics['roc_auc']:.4f} | " f"Val Profit: {valid_metrics['profit']}")
        
        
        logging.info(
            f"Эпоха {epoch} | "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Val ROC-AUC: {valid_metrics['roc_auc']:.4f} | "
            f"Val Profit: {valid_metrics['profit']}"
        )

        wandb.log({
             f"{model_name}/train_loss": epoch_loss,
             f"{model_name}/valid_profit": valid_metrics["profit"],
             f"{model_name}/valid_roc_auc": valid_metrics["roc_auc"],
             f"{model_name}/valid_precision": valid_metrics["precision"],
             f"{model_name}/valid_recall": valid_metrics["recall"],
             f"{model_name}/valid_f05": valid_metrics["f05"],
         })

    model.load_state_dict(best_state)

    print()
    print(f"Лучшая эпоха для {model_name}: {best_epoch}")
    print(f"Лучший ROC-AUC: {best_metric}")
    logging.info("Закончили обучение")
    logging.info(f"Лучшая эпоха для {model_name}: {best_epoch}")
    logging.info(f"Лучший ROC-AUC: {best_metric}")

    return model

In [25]:
def save_results(model, name, log_file, run):
    train_metrics = evaluate_model(model, X_train, y_train, threshold=0.5, name="Train")
    test_metrics = evaluate_model(model, X_test, y_test, threshold=0.5, name="Test")
    pickle.dump(model.state_dict(), open(f"models/{name}.pkl", 'wb'))
    logging.info("Сохранили веса модели в папку models")

    metric_keys = list(train_metrics.keys())

    table = wandb.Table(columns=metric_keys)
    table.add_data(*[train_metrics[i] for i in metric_keys])
    table.add_data(*[test_metrics[i] for i in metric_keys])
    wandb.log({'results': table})
    artifact = wandb.Artifact(name=name, type="model", description=f"Тест логгирования модели: {name}")

    artifact.add_file(f"models/{name}.pkl")
    artifact.add_file(log_file)
    run.log_artifact(artifact) 

## model_1_baseline

In [26]:
EPOCHS = 30
SEED = 42

Отключаем логирование на переборе гиперпараметров

In [27]:
wandb.init(mode="disabled")
results = []
for LR in [0.05, 0.03, 0.01, 0.005, 0.002, 0.0001]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_1 = nn.Sequential(
            nn.Linear(9, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_1.parameters(), lr=LR)

        model_1 = train_model(
            model=model_1,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_1_baseline"
        )

        val = evaluate_model(model_1, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_1_baseline | valid epoch 1
ROC-AUC: 0.9363
Precision: 0.2769
Recall: 0.8571
F0.5-score: 0.3203
Profit: -1100

Эпоха 1 | Train Loss: 0.8771 | Val ROC-AUC: 0.9363 | Val Profit: -1100
model_1_baseline | valid epoch 2
ROC-AUC: 0.9608
Precision: 0.2019
Recall: 1.0
F0.5-score: 0.2403
Profit: -1970

Эпоха 2 | Train Loss: 0.6035 | Val ROC-AUC: 0.9608 | Val Profit: -1970
model_1_baseline | valid epoch 3
ROC-AUC: 0.972
Precision: 0.2283
Recall: 1.0
F0.5-score: 0.2699
Profit: -1670

Эпоха 3 | Train Loss: 0.4833 | Val ROC-AUC: 0.9720 | Val Profit: -1670
model_1_baseline | valid epoch 4
ROC-AUC: 0.9714
Precision: 0.3281
Recall: 1.0
F0.5-score: 0.3791
Profit: -970

Эпоха 4 | Train Loss: 0.3636 | Val ROC-AUC: 0.9714 | Val Profit: -970
model_1_baseline | valid epoch 5
ROC-AUC: 0.9771
Precision: 0.3443
Recall: 1.0
F0.5-score: 0.3962
Profit: -895

Эпоха 5 | Train Loss: 0.3314 | Val ROC-AUC: 0.9771 | Val Profit: -895
model_1_baseline | valid epoch 6
ROC-AUC: 0.9685
Precision: 0.3387
Recall: 1.0
F0.

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_1_baseline | valid epoch 14
ROC-AUC: 0.9801
Precision: 0.375
Recall: 1.0
F0.5-score: 0.4286
Profit: -770

Эпоха 14 | Train Loss: 0.2125 | Val ROC-AUC: 0.9801 | Val Profit: -770
model_1_baseline | valid epoch 15
ROC-AUC: 0.9804
Precision: 0.3818
Recall: 1.0
F0.5-score: 0.4357
Profit: -745

Эпоха 15 | Train Loss: 0.2021 | Val ROC-AUC: 0.9804 | Val Profit: -745
model_1_baseline | valid epoch 16
ROC-AUC: 0.9815
Precision: 0.3962
Recall: 1.0
F0.5-score: 0.4506
Profit: -695

Эпоха 16 | Train Loss: 0.1917 | Val ROC-AUC: 0.9815 | Val Profit: -695
model_1_baseline | valid epoch 17
ROC-AUC: 0.9822
Precision: 0.4375
Recall: 1.0
F0.5-score: 0.493
Profit: -570

Эпоха 17 | Train Loss: 0.1817 | Val ROC-AUC: 0.9822 | Val Profit: -570
model_1_baseline | valid epoch 18
ROC-AUC: 0.9836
Precision: 0.4118
Recall: 1.0
F0.5-score: 0.4667
Profit: -645

Эпоха 18 | Train Loss: 0.1746 | Val ROC-AUC: 0.9836 | Val Profit: -645
model_1_baseline | valid epoch 19
ROC-AUC: 0.984
Precision: 0.4667
Recall: 1.0
F0.

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_1_baseline | valid epoch 5
ROC-AUC: 0.9477
Precision: 0.28
Recall: 1.0
F0.5-score: 0.3271
Profit: -1245

Эпоха 5 | Train Loss: 0.8859 | Val ROC-AUC: 0.9477 | Val Profit: -1245
model_1_baseline | valid epoch 6
ROC-AUC: 0.9539
Precision: 0.2471
Recall: 1.0
F0.5-score: 0.2909
Profit: -1495

Эпоха 6 | Train Loss: 0.7036 | Val ROC-AUC: 0.9539 | Val Profit: -1495
model_1_baseline | valid epoch 7
ROC-AUC: 0.9591
Precision: 0.2471
Recall: 1.0
F0.5-score: 0.2909
Profit: -1495

Эпоха 7 | Train Loss: 0.5755 | Val ROC-AUC: 0.9591 | Val Profit: -1495
model_1_baseline | valid epoch 8
ROC-AUC: 0.9622
Precision: 0.25
Recall: 1.0
F0.5-score: 0.2941
Profit: -1470

Эпоха 8 | Train Loss: 0.4992 | Val ROC-AUC: 0.9622 | Val Profit: -1470
model_1_baseline | valid epoch 9
ROC-AUC: 0.9657
Precision: 0.2561
Recall: 1.0
F0.5-score: 0.3009
Profit: -1420

Эпоха 9 | Train Loss: 0.4507 | Val ROC-AUC: 0.9657 | Val Profit: -1420
model_1_baseline | valid epoch 10
ROC-AUC: 0.9677
Precision: 0.2561
Recall: 1.0
F0.5

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_1_baseline | valid epoch 17
ROC-AUC: 0.8534
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 17 | Train Loss: 1.3050 | Val ROC-AUC: 0.8534 | Val Profit: -105
model_1_baseline | valid epoch 18
ROC-AUC: 0.8565
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 18 | Train Loss: 1.3029 | Val ROC-AUC: 0.8565 | Val Profit: -105
model_1_baseline | valid epoch 19
ROC-AUC: 0.8596
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 19 | Train Loss: 1.3007 | Val ROC-AUC: 0.8596 | Val Profit: -105
model_1_baseline | valid epoch 20
ROC-AUC: 0.8633
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 20 | Train Loss: 1.2985 | Val ROC-AUC: 0.8633 | Val Profit: -105
model_1_baseline | valid epoch 21
ROC-AUC: 0.8611
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 21 | Train Loss: 1.2962 | Val ROC-AUC: 0.8611 | Val Profit: -105
model_1_baseline | valid epoch 22
ROC-AUC: 0.8636
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпо

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

In [28]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.0001, 'profit': np.int64(-105), 'roc_auc': 0.8847753185781355}


In [29]:
LR = best["lr"]

In [30]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

model_1 = nn.Sequential(
    nn.Linear(9, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_1.parameters(), lr=LR)
config = {
    "model": "MLP",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_1)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_1_baseline", config=config)
log_file = new_log_file()

model_1 = train_model(
    model=model_1,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_1_baseline"
)

save_results(model_1, "model_1_baseline", log_file, run)

run.finish()

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_1_baseline | valid epoch 1
ROC-AUC: 0.6702
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 1 | Train Loss: 1.3347 | Val ROC-AUC: 0.6702 | Val Profit: -105
model_1_baseline | valid epoch 2
ROC-AUC: 0.6936
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 2 | Train Loss: 1.3326 | Val ROC-AUC: 0.6936 | Val Profit: -105
model_1_baseline | valid epoch 3
ROC-AUC: 0.7123
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 3 | Train Loss: 1.3305 | Val ROC-AUC: 0.7123 | Val Profit: -105
model_1_baseline | valid epoch 4
ROC-AUC: 0.7319
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 4 | Train Loss: 1.3286 | Val ROC-AUC: 0.7319 | Val Profit: -105
model_1_baseline | valid epoch 5
ROC-AUC: 0.7497
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 5 | Train Loss: 1.3267 | Val ROC-AUC: 0.7497 | Val Profit: -105
model_1_baseline | valid epoch 6
ROC-AUC: 0.7667
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 6 | Trai

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_1_baseline | valid epoch 20
ROC-AUC: 0.8633
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 20 | Train Loss: 1.2985 | Val ROC-AUC: 0.8633 | Val Profit: -105
model_1_baseline | valid epoch 21
ROC-AUC: 0.8611
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 21 | Train Loss: 1.2962 | Val ROC-AUC: 0.8611 | Val Profit: -105
model_1_baseline | valid epoch 22
ROC-AUC: 0.8636
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 22 | Train Loss: 1.2938 | Val ROC-AUC: 0.8636 | Val Profit: -105
model_1_baseline | valid epoch 23
ROC-AUC: 0.8691
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 23 | Train Loss: 1.2913 | Val ROC-AUC: 0.8691 | Val Profit: -105
model_1_baseline | valid epoch 24
ROC-AUC: 0.8719
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 24 | Train Loss: 1.2887 | Val ROC-AUC: 0.8719 | Val Profit: -105
model_1_baseline | valid epoch 25
ROC-AUC: 0.8735
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпо

model_1_baseline/train_loss,███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▂▂▂▁▁
model_1_baseline/valid_f05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_precision,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_profit,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_recall,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_roc_auc,▁▂▂▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████
model_1_baseline/train_loss,1.27086
model_1_baseline/valid_f05,0
model_1_baseline/valid_precision,0
model_1_baseline/valid_profit,-105
model_1_baseline/valid_recall,0


In [31]:
train_metrics = evaluate_model(model_1, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_1, X_test, y_test, threshold=0.5, name="Test")

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Train
ROC-AUC: 0.9065
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -415

Test
ROC-AUC: 0.9032
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -118635



Видно предупреждение в выводе о том, что модель не предсказала ни одного фрода. Получилась недообученная модель. Исходя из функции profit был выбран самый маленький lr, и модель с ним не успела обучиться

## model_2_dop_sloi

In [32]:
EPOCHS = 30
SEED = 42

In [33]:
wandb.init(mode="disabled")
results = []
for LR in [0.05, 0.03, 0.01, 0.005, 0.002, 0.0001]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_2 = nn.Sequential(
            nn.Linear(9, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_2.parameters(), lr=LR)

        model_2 = train_model(
            model=model_2,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_2_dop_sloi"
        )

        val = evaluate_model(model_2, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_2_dop_sloi | valid epoch 1
ROC-AUC: 0.9133
Precision: 0.2346
Recall: 0.9048
F0.5-score: 0.2754
Profit: -1465

Эпоха 1 | Train Loss: 1.0974 | Val ROC-AUC: 0.9133 | Val Profit: -1465
model_2_dop_sloi | valid epoch 2
ROC-AUC: 0.9355
Precision: 0.2879
Recall: 0.9048
F0.5-score: 0.3333
Profit: -1090

Эпоха 2 | Train Loss: 0.6756 | Val ROC-AUC: 0.9355 | Val Profit: -1090
model_2_dop_sloi | valid epoch 3
ROC-AUC: 0.9292
Precision: 0.2471
Recall: 1.0
F0.5-score: 0.2909
Profit: -1495

Эпоха 3 | Train Loss: 0.5462 | Val ROC-AUC: 0.9292 | Val Profit: -1495
model_2_dop_sloi | valid epoch 4
ROC-AUC: 0.9064
Precision: 0.2211
Recall: 1.0
F0.5-score: 0.2618
Profit: -1745

Эпоха 4 | Train Loss: 0.4843 | Val ROC-AUC: 0.9064 | Val Profit: -1745
model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.862
Precision: 0.1544
Recall: 1.0
F0.5-score: 0.1858
Profit: -2770

Эпоха 5 | Train Loss: 0.5720 | Val ROC-AUC: 0.8620 | Val Profit: -2770
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.869
Precision: 0.1556
Recall: 1

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.9616
Precision: 0.2727
Recall: 1.0
F0.5-score: 0.3191
Profit: -1295

Эпоха 5 | Train Loss: 0.3485 | Val ROC-AUC: 0.9616 | Val Profit: -1295
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.9544
Precision: 0.2763
Recall: 1.0
F0.5-score: 0.3231
Profit: -1270

Эпоха 6 | Train Loss: 0.3119 | Val ROC-AUC: 0.9544 | Val Profit: -1270
model_2_dop_sloi | valid epoch 7
ROC-AUC: 0.9531
Precision: 0.2658
Recall: 1.0
F0.5-score: 0.3116
Profit: -1345

Эпоха 7 | Train Loss: 0.2901 | Val ROC-AUC: 0.9531 | Val Profit: -1345
model_2_dop_sloi | valid epoch 8
ROC-AUC: 0.9612
Precision: 0.3
Recall: 1.0
F0.5-score: 0.3488
Profit: -1120

Эпоха 8 | Train Loss: 0.2719 | Val ROC-AUC: 0.9612 | Val Profit: -1120
model_2_dop_sloi | valid epoch 9
ROC-AUC: 0.9646
Precision: 0.3571
Recall: 0.9524
F0.5-score: 0.4082
Profit: -805

Эпоха 9 | Train Loss: 0.2528 | Val ROC-AUC: 0.9646 | Val Profit: -805
model_2_dop_sloi | valid epoch 10
ROC-AUC: 0.9559
Precision: 0.3654
Recall: 0.9048

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_2_dop_sloi | valid epoch 12
ROC-AUC: 0.9666
Precision: 0.3509
Recall: 0.9524
F0.5-score: 0.4016
Profit: -830

Эпоха 12 | Train Loss: 0.2427 | Val ROC-AUC: 0.9666 | Val Profit: -830
model_2_dop_sloi | valid epoch 13
ROC-AUC: 0.9678
Precision: 0.3585
Recall: 0.9048
F0.5-score: 0.4077
Profit: -765

Эпоха 13 | Train Loss: 0.2258 | Val ROC-AUC: 0.9678 | Val Profit: -765
model_2_dop_sloi | valid epoch 14
ROC-AUC: 0.9689
Precision: 0.3654
Recall: 0.9048
F0.5-score: 0.4148
Profit: -740

Эпоха 14 | Train Loss: 0.2121 | Val ROC-AUC: 0.9689 | Val Profit: -740
model_2_dop_sloi | valid epoch 15
ROC-AUC: 0.969
Precision: 0.3725
Recall: 0.9048
F0.5-score: 0.4222
Profit: -715

Эпоха 15 | Train Loss: 0.1986 | Val ROC-AUC: 0.9690 | Val Profit: -715
model_2_dop_sloi | valid epoch 16
ROC-AUC: 0.97
Precision: 0.38
Recall: 0.9048
F0.5-score: 0.4299
Profit: -690

Эпоха 16 | Train Loss: 0.1864 | Val ROC-AUC: 0.9700 | Val Profit: -690
model_2_dop_sloi | valid epoch 17
ROC-AUC: 0.9704
Precision: 0.413
Rec

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

Эпоха 11 | Train Loss: 1.2912 | Val ROC-AUC: 0.8731 | Val Profit: -105
model_2_dop_sloi | valid epoch 12
ROC-AUC: 0.8845
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 12 | Train Loss: 1.2874 | Val ROC-AUC: 0.8845 | Val Profit: -105
model_2_dop_sloi | valid epoch 13
ROC-AUC: 0.8956
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 13 | Train Loss: 1.2832 | Val ROC-AUC: 0.8956 | Val Profit: -105
model_2_dop_sloi | valid epoch 14
ROC-AUC: 0.904
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 14 | Train Loss: 1.2785 | Val ROC-AUC: 0.9040 | Val Profit: -105
model_2_dop_sloi | valid epoch 15
ROC-AUC: 0.9113
Precision: 1.0
Recall: 0.0476
F0.5-score: 0.2
Profit: -95

Эпоха 15 | Train Loss: 1.2731 | Val ROC-AUC: 0.9113 | Val Profit: -95
model_2_dop_sloi | valid epoch 16
ROC-AUC: 0.9172
Precision: 0.6
Recall: 0.1429
F0.5-score: 0.3659
Profit: -125

Эпоха 16 | Train Loss: 1.2671 | Val ROC-AUC: 0.9172 | Val Profit: -125
model_2_dop_sloi | valid epoch 17

In [34]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.01, 'profit': np.int64(-150), 'roc_auc': 0.9884641180415827}


In [35]:
LR = best["lr"]

In [36]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_2 = nn.Sequential(
        nn.Linear(9, 32),
        nn.ReLU(),

        nn.Linear(32, 16),
        nn.ReLU(),

        nn.Linear(16, 8),
        nn.ReLU(),

        nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_2.parameters(), lr=LR)

config = {
    "model": "MLP_added_layer",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_2)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_2_added_layers", config=config)
log_file = new_log_file()

model_2 = train_model(
    model=model_2,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_2_dop_sloi"
)

save_results(model_2, "model_2", log_file, run)

run.finish()

model_2_dop_sloi | valid epoch 1
ROC-AUC: 0.9311
Precision: 0.3051
Recall: 0.8571
F0.5-score: 0.3502
Profit: -950

Эпоха 1 | Train Loss: 1.1825 | Val ROC-AUC: 0.9311 | Val Profit: -950
model_2_dop_sloi | valid epoch 2
ROC-AUC: 0.9552
Precision: 0.3
Recall: 1.0
F0.5-score: 0.3488
Profit: -1120

Эпоха 2 | Train Loss: 0.6270 | Val ROC-AUC: 0.9552 | Val Profit: -1120
model_2_dop_sloi | valid epoch 3
ROC-AUC: 0.9502
Precision: 0.25
Recall: 1.0
F0.5-score: 0.2941
Profit: -1470

Эпоха 3 | Train Loss: 0.4225 | Val ROC-AUC: 0.9502 | Val Profit: -1470
model_2_dop_sloi | valid epoch 4
ROC-AUC: 0.9642
Precision: 0.2857
Recall: 0.9524
F0.5-score: 0.3322
Profit: -1155

Эпоха 4 | Train Loss: 0.4105 | Val ROC-AUC: 0.9642 | Val Profit: -1155
model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.9641
Precision: 0.3182
Recall: 1.0
F0.5-score: 0.3684
Profit: -1020

Эпоха 5 | Train Loss: 0.3026 | Val ROC-AUC: 0.9641 | Val Profit: -1020
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.9599
Precision: 0.2941
Recall: 0.9524

IOStream.flush timed out
IOStream.flush timed out


model_2_dop_sloi/train_loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▁▁▃▂▁▁▁▁▁▂▁
model_2_dop_sloi/valid_f05,▂▂▁▂▂▂▃▃▄▃▃▃▄▄▄▆▆▆▆▄▅▂▄▄▇█▅█▆▆
model_2_dop_sloi/valid_precision,▂▂▁▂▂▂▃▃▄▃▃▃▄▄▄▆▅▆▆▃▅▂▄▄▆█▅█▆▆
model_2_dop_sloi/valid_profit,▄▃▁▃▃▃▄▅▆▅▅▅▆▆▆▇▇▇▇▅▇▃▆▆▇█▆█▇▇
model_2_dop_sloi/valid_recall,▄██▇█▇▇▇▁█▅█▇▄▅▅▅▅▄█▂███▅▄█▄▄▅
model_2_dop_sloi/valid_roc_auc,▁▄▃▅▅▅▅▅▅▅▅▆▇▆▆▇▇▇▇▆▆▅▆▇████▇█
model_2_dop_sloi/train_loss,0.15347
model_2_dop_sloi/valid_f05,0.6051
model_2_dop_sloi/valid_precision,0.55882
model_2_dop_sloi/valid_profit,-290
model_2_dop_sloi/valid_recall,0.90476


In [37]:
train_metrics = evaluate_model(model_2, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_2, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9974
Precision: 0.7345
Recall: 1.0
F0.5-score: 0.7757
Profit: -335

Test
ROC-AUC: 0.9854
Precision: 0.5884
Recall: 0.8528
F0.5-score: 0.6273
Profit: -270185



Да, добавив всего 1 доп слой, видно, что модель стала находить зависимости. Precision и Recall больше не равны нулю. Profit равен -270185 (что однако хуже бейзлайна без предсказания фрода)

## model_3_bolshe_neyronov

In [38]:
EPOCHS = 30
SEED = 42

In [39]:
wandb.init(mode="disabled")
results = []
for LR in [0.05, 0.03, 0.01, 0.005, 0.002, 0.0001]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_3 = nn.Sequential(
            nn.Linear(9, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_3.parameters(), lr=LR)

        model_3 = train_model(
            model=model_3,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_3_bolshe_neyronov"
        )

        val = evaluate_model(model_3, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_3_bolshe_neyronov | valid epoch 1
ROC-AUC: 0.9429
Precision: 0.2727
Recall: 0.8571
F0.5-score: 0.3158
Profit: -1125

Эпоха 1 | Train Loss: 0.8665 | Val ROC-AUC: 0.9429 | Val Profit: -1125
model_3_bolshe_neyronov | valid epoch 2
ROC-AUC: 0.9569
Precision: 0.253
Recall: 1.0
F0.5-score: 0.2975
Profit: -1445

Эпоха 2 | Train Loss: 0.5109 | Val ROC-AUC: 0.9569 | Val Profit: -1445
model_3_bolshe_neyronov | valid epoch 3
ROC-AUC: 0.9693
Precision: 0.3509
Recall: 0.9524
F0.5-score: 0.4016
Profit: -830

Эпоха 3 | Train Loss: 0.4088 | Val ROC-AUC: 0.9693 | Val Profit: -830
model_3_bolshe_neyronov | valid epoch 4
ROC-AUC: 0.9698
Precision: 0.3922
Recall: 0.9524
F0.5-score: 0.4444
Profit: -680

Эпоха 4 | Train Loss: 0.3122 | Val ROC-AUC: 0.9698 | Val Profit: -680
model_3_bolshe_neyronov | valid epoch 5
ROC-AUC: 0.9689
Precision: 0.3443
Recall: 1.0
F0.5-score: 0.3962
Profit: -895

Эпоха 5 | Train Loss: 0.2802 | Val ROC-AUC: 0.9689 | Val Profit: -895
model_3_bolshe_neyronov | valid epoch 6
ROC

In [40]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.005, 'profit': np.int64(-150), 'roc_auc': 0.984842387659289}


In [41]:
LR = best["lr"]

In [42]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

model_3 = nn.Sequential(
    nn.Linear(9, 32),
    nn.ReLU(),
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_3.parameters(), lr=LR)

config = {
    "model": "MLP_added_neurons",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_3)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_3_more_neurons", config=config)
log_file = new_log_file()

model_3 = train_model(
    model=model_3,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_3_bolshe_neyronov"
)


save_results(model_3, "model_3", log_file, run)

run.finish()

model_3_bolshe_neyronov | valid epoch 1
ROC-AUC: 0.9571
Precision: 0.1615
Recall: 1.0
F0.5-score: 0.1941
Profit: -2620

Эпоха 1 | Train Loss: 1.0990 | Val ROC-AUC: 0.9571 | Val Profit: -2620
model_3_bolshe_neyronov | valid epoch 2
ROC-AUC: 0.9622
Precision: 0.2234
Recall: 1.0
F0.5-score: 0.2645
Profit: -1720

Эпоха 2 | Train Loss: 0.5891 | Val ROC-AUC: 0.9622 | Val Profit: -1720
model_3_bolshe_neyronov | valid epoch 3
ROC-AUC: 0.9622
Precision: 0.2658
Recall: 1.0
F0.5-score: 0.3116
Profit: -1345

Эпоха 3 | Train Loss: 0.4419 | Val ROC-AUC: 0.9622 | Val Profit: -1345
model_3_bolshe_neyronov | valid epoch 4
ROC-AUC: 0.9653
Precision: 0.2561
Recall: 1.0
F0.5-score: 0.3009
Profit: -1420

Эпоха 4 | Train Loss: 0.3847 | Val ROC-AUC: 0.9653 | Val Profit: -1420
model_3_bolshe_neyronov | valid epoch 5
ROC-AUC: 0.9674
Precision: 0.2692
Recall: 1.0
F0.5-score: 0.3153
Profit: -1320

Эпоха 5 | Train Loss: 0.3440 | Val ROC-AUC: 0.9674 | Val Profit: -1320
model_3_bolshe_neyronov | valid epoch 6
ROC-A

model_3_bolshe_neyronov/train_loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_3_bolshe_neyronov/valid_f05,▁▂▃▂▃▃▃▃▄▄▄▅▅▅▅▅▆▆▇▇█▇▇▇█▆▇▆▆▇
model_3_bolshe_neyronov/valid_precision,▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▅▆▆▇▇█▇▇▇█▆▇▆▆▇
model_3_bolshe_neyronov/valid_profit,▁▄▅▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇███████▇█▇▇█
model_3_bolshe_neyronov/valid_recall,████████▆▆▆▆▆▆▆▆▆▆▆▃▃▃▃▁▁▃▃▃▃▆
model_3_bolshe_neyronov/valid_roc_auc,▁▂▂▃▄▃▃▄▄▄▅▆▆▆▇▇▇▇▇██▇▇██▇▇▆▆█
model_3_bolshe_neyronov/train_loss,0.1125
model_3_bolshe_neyronov/valid_f05,0.62112
model_3_bolshe_neyronov/valid_precision,0.57143
model_3_bolshe_neyronov/valid_profit,-280
model_3_bolshe_neyronov/valid_recall,0.95238


In [43]:
train_metrics = evaluate_model(model_3, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_3, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9996
Precision: 0.8737
Recall: 1.0
F0.5-score: 0.8963
Profit: 115

Test
ROC-AUC: 0.9861
Precision: 0.6325
Recall: 0.8125
F0.5-score: 0.6618
Profit: -205930



Увеличение количества нейронов в 2 раза улучшило метрики ROC-AUC, Precision и Recall. Однако profit все еще хуже бейзлайна 

## model_4_tolko_batchnorm

In [44]:
EPOCHS = 30
SEED = 42

In [45]:
wandb.init(mode="disabled")
results = []
for LR in [0.05, 0.03, 0.01, 0.005, 0.002, 0.0001]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_4 = nn.Sequential(
            nn.Linear(9, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.BatchNorm1d(8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_4.parameters(), lr=LR)

        model_4 = train_model(
            model=model_4,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_4_tolko_batchnorm"
        )

        val = evaluate_model(model_4, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_4_tolko_batchnorm | valid epoch 1
ROC-AUC: 0.9757
Precision: 0.1419
Recall: 1.0
F0.5-score: 0.1713
Profit: -3070

Эпоха 1 | Train Loss: 0.9045 | Val ROC-AUC: 0.9757 | Val Profit: -3070
model_4_tolko_batchnorm | valid epoch 2
ROC-AUC: 0.9631
Precision: 0.2258
Recall: 1.0
F0.5-score: 0.2672
Profit: -1695

Эпоха 2 | Train Loss: 0.4640 | Val ROC-AUC: 0.9631 | Val Profit: -1695
model_4_tolko_batchnorm | valid epoch 3
ROC-AUC: 0.9547
Precision: 0.2059
Recall: 1.0
F0.5-score: 0.2448
Profit: -1920

Эпоха 3 | Train Loss: 0.4584 | Val ROC-AUC: 0.9547 | Val Profit: -1920
model_4_tolko_batchnorm | valid epoch 4
ROC-AUC: 0.9733
Precision: 0.4722
Recall: 0.8095
F0.5-score: 0.5152
Profit: -410

Эпоха 4 | Train Loss: 0.3655 | Val ROC-AUC: 0.9733 | Val Profit: -410
model_4_tolko_batchnorm | valid epoch 5
ROC-AUC: 0.951
Precision: 0.2958
Recall: 1.0
F0.5-score: 0.3443
Profit: -1145

Эпоха 5 | Train Loss: 0.3121 | Val ROC-AUC: 0.9510 | Val Profit: -1145
model_4_tolko_batchnorm | valid epoch 6
ROC-A

In [46]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.05, 'profit': np.int64(-25), 'roc_auc': 0.986317907444668}


In [47]:
LR = best["lr"]

In [48]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_4 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),

    nn.Linear(8, 1)
)

config = {
    "model": "MLP_batchnorm_only",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_4)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_4_batchnorm_only", config=config)
log_file = new_log_file()

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_4.parameters(), lr=LR)

model_4 = train_model(
    model=model_4,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_4_tolko_batchnorm"
)


save_results(model_4, "model_4", log_file, run)

run.finish()

model_4_tolko_batchnorm | valid epoch 1
ROC-AUC: 0.9757
Precision: 0.1419
Recall: 1.0
F0.5-score: 0.1713
Profit: -3070

Эпоха 1 | Train Loss: 0.9045 | Val ROC-AUC: 0.9757 | Val Profit: -3070
model_4_tolko_batchnorm | valid epoch 2
ROC-AUC: 0.9631
Precision: 0.2258
Recall: 1.0
F0.5-score: 0.2672
Profit: -1695

Эпоха 2 | Train Loss: 0.4640 | Val ROC-AUC: 0.9631 | Val Profit: -1695
model_4_tolko_batchnorm | valid epoch 3
ROC-AUC: 0.9547
Precision: 0.2059
Recall: 1.0
F0.5-score: 0.2448
Profit: -1920

Эпоха 3 | Train Loss: 0.4584 | Val ROC-AUC: 0.9547 | Val Profit: -1920
model_4_tolko_batchnorm | valid epoch 4
ROC-AUC: 0.9733
Precision: 0.4722
Recall: 0.8095
F0.5-score: 0.5152
Profit: -410

Эпоха 4 | Train Loss: 0.3655 | Val ROC-AUC: 0.9733 | Val Profit: -410
model_4_tolko_batchnorm | valid epoch 5
ROC-AUC: 0.951
Precision: 0.2958
Recall: 1.0
F0.5-score: 0.3443
Profit: -1145

Эпоха 5 | Train Loss: 0.3121 | Val ROC-AUC: 0.9510 | Val Profit: -1145
model_4_tolko_batchnorm | valid epoch 6
ROC-A

model_4_tolko_batchnorm/train_loss,█▄▄▃▃▄▃▃▂▂▂▂▂▂▂▂▁▃▂▂▁▂▂▂▂▁▁▂▂▁
model_4_tolko_batchnorm/valid_f05,▂▃▃▅▄▃▆▆▆▃▅▅▅▆▅▅▆▃▆▆▅▆▆▇▅▁▅▆█▄
model_4_tolko_batchnorm/valid_precision,▂▃▃▅▃▃▅▅▅▅▅▇▄▆▅▅▇▃▆▇▇▅▅▇▇▁▆▅█▆
model_4_tolko_batchnorm/valid_profit,▁▄▄▇▅▄▇▇▇█▇█▆▇▇▇█▄▇██▇▇████▇██
model_4_tolko_batchnorm/valid_recall,███▇██▆█▆▁▇▃█▇▇▇▃█▆▄▂██▅▂▁▂▇▅▂
model_4_tolko_batchnorm/valid_roc_auc,▆▃▂▅▁▆▅▆▇▆▆▅▅▇▅▆▇▇▇▇▆▇▇▇▇▇█▅█▆
model_4_tolko_batchnorm/train_loss,0.1718
model_4_tolko_batchnorm/valid_f05,0.30303
model_4_tolko_batchnorm/valid_precision,0.66667
model_4_tolko_batchnorm/valid_profit,-110
model_4_tolko_batchnorm/valid_recall,0.09524


In [49]:
train_metrics = evaluate_model(model_4, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_4, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9926
Precision: 0.8814
Recall: 0.6265
F0.5-score: 0.815
Profit: -70

Test
ROC-AUC: 0.9825
Precision: 0.7598
Recall: 0.5193
F0.5-score: 0.6954
Profit: -92825



Так как в предыдущие разы качество улучшилось с увеличением колво слоев, то теперь мы решили добавить колво слоев, а также после каждого слоя сделать только BatchNorm

Видно, что это сильно улучшило качество! И мы побили бейзлайн!

## model_5_tolko_dropout

In [50]:
EPOCHS = 40
SEED = 42

In [51]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005, 0.001]:
    for DROPOUT_COEF in [0.05, 0.1, 0.015, 0.2, 0.025, 0.3]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_5 = nn.Sequential(
            nn.Linear(9, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(8, 1)
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_5.parameters(), lr=LR)
        model_5 = train_model(
            model=model_5,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_5_tolko_dropout"
        )

        val = evaluate_model(model_5, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "dropout": DROPOUT_COEF, "profit": val["profit"], "roc_auc": val["roc_auc"]})


model_5_tolko_dropout | valid epoch 1
ROC-AUC: 0.9353
Precision: 0.2414
Recall: 1.0
F0.5-score: 0.2846
Profit: -1545

Эпоха 1 | Train Loss: 1.0889 | Val ROC-AUC: 0.9353 | Val Profit: -1545
model_5_tolko_dropout | valid epoch 2
ROC-AUC: 0.9204
Precision: 0.2121
Recall: 1.0
F0.5-score: 0.2518
Profit: -1845

Эпоха 2 | Train Loss: 0.8168 | Val ROC-AUC: 0.9204 | Val Profit: -1845
model_5_tolko_dropout | valid epoch 3
ROC-AUC: 0.9276
Precision: 0.2923
Recall: 0.9048
F0.5-score: 0.3381
Profit: -1065

Эпоха 3 | Train Loss: 0.6146 | Val ROC-AUC: 0.9276 | Val Profit: -1065
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9619
Precision: 0.3846
Recall: 0.9524
F0.5-score: 0.4367
Profit: -705

Эпоха 4 | Train Loss: 0.4525 | Val ROC-AUC: 0.9619 | Val Profit: -705
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9322
Precision: 0.274
Recall: 0.9524
F0.5-score: 0.3195
Profit: -1230

Эпоха 5 | Train Loss: 0.4916 | Val ROC-AUC: 0.9322 | Val Profit: -1230
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_5_tolko_dropout | valid epoch 10
ROC-AUC: 0.9686
Precision: 0.3922
Recall: 0.9524
F0.5-score: 0.4444
Profit: -680

Эпоха 10 | Train Loss: 0.2376 | Val ROC-AUC: 0.9686 | Val Profit: -680
model_5_tolko_dropout | valid epoch 11
ROC-AUC: 0.9596
Precision: 0.2923
Recall: 0.9048
F0.5-score: 0.3381
Profit: -1065

Эпоха 11 | Train Loss: 0.2943 | Val ROC-AUC: 0.9596 | Val Profit: -1065
model_5_tolko_dropout | valid epoch 12
ROC-AUC: 0.9591
Precision: 0.2561
Recall: 1.0
F0.5-score: 0.3009
Profit: -1420

Эпоха 12 | Train Loss: 0.3189 | Val ROC-AUC: 0.9591 | Val Profit: -1420
model_5_tolko_dropout | valid epoch 13
ROC-AUC: 0.9651
Precision: 0.3774
Recall: 0.9524
F0.5-score: 0.4292
Profit: -730

Эпоха 13 | Train Loss: 0.2749 | Val ROC-AUC: 0.9651 | Val Profit: -730
model_5_tolko_dropout | valid epoch 14
ROC-AUC: 0.9565
Precision: 0.3774
Recall: 0.9524
F0.5-score: 0.4292
Profit: -730

Эпоха 14 | Train Loss: 0.1901 | Val ROC-AUC: 0.9565 | Val Profit: -730
model_5_tolko_dropout | valid epoch 15


/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_5_tolko_dropout | valid epoch 12
ROC-AUC: 0.9623
Precision: 0.339
Recall: 0.9524
F0.5-score: 0.3891
Profit: -880

Эпоха 12 | Train Loss: 0.2890 | Val ROC-AUC: 0.9623 | Val Profit: -880
model_5_tolko_dropout | valid epoch 13
ROC-AUC: 0.9608
Precision: 0.3226
Recall: 0.9524
F0.5-score: 0.3717
Profit: -955

Эпоха 13 | Train Loss: 0.2659 | Val ROC-AUC: 0.9608 | Val Profit: -955
model_5_tolko_dropout | valid epoch 14
ROC-AUC: 0.9619
Precision: 0.3393
Recall: 0.9048
F0.5-score: 0.3878
Profit: -840

Эпоха 14 | Train Loss: 0.2377 | Val ROC-AUC: 0.9619 | Val Profit: -840
model_5_tolko_dropout | valid epoch 15
ROC-AUC: 0.9586
Precision: 0.3115
Recall: 0.9048
F0.5-score: 0.3585
Profit: -965

Эпоха 15 | Train Loss: 0.2238 | Val ROC-AUC: 0.9586 | Val Profit: -965
model_5_tolko_dropout | valid epoch 16
ROC-AUC: 0.9618
Precision: 0.3279
Recall: 0.9524
F0.5-score: 0.3774
Profit: -930

Эпоха 16 | Train Loss: 0.2517 | Val ROC-AUC: 0.9618 | Val Profit: -930
model_5_tolko_dropout | valid epoch 17
RO

...

model_5_tolko_dropout | valid epoch 2
ROC-AUC: 0.9502
Precision: 0.4571
Recall: 0.7619
F0.5-score: 0.4969
Profit: -420

Эпоха 2 | Train Loss: 1.2790 | Val ROC-AUC: 0.9502 | Val Profit: -420
model_5_tolko_dropout | valid epoch 3
ROC-AUC: 0.9485
Precision: 0.2923
Recall: 0.9048
F0.5-score: 0.3381
Profit: -1065

Эпоха 3 | Train Loss: 1.1307 | Val ROC-AUC: 0.9485 | Val Profit: -1065
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9427
Precision: 0.2692
Recall: 1.0
F0.5-score: 0.3153
Profit: -1320

Эпоха 4 | Train Loss: 0.8866 | Val ROC-AUC: 0.9427 | Val Profit: -1320
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9508
Precision: 0.2838
Recall: 1.0
F0.5-score: 0.3312
Profit: -1220

Эпоха 5 | Train Loss: 0.6292 | Val ROC-AUC: 0.9508 | Val Profit: -1220
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.9522
Precision: 0.2917
Recall: 1.0
F0.5-score: 0.3398
Profit: -1170

Эпоха 6 | Train Loss: 0.4478 | Val ROC-AUC: 0.9522 | Val Profit: -1170
model_5_tolko_dropout | valid epoch 7
ROC-AUC: 0.96

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Эпоха 3 | Train Loss: 1.0387 | Val ROC-AUC: 0.9521 | Val Profit: -1130
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9317
Precision: 0.253
Recall: 1.0
F0.5-score: 0.2975
Profit: -1445

Эпоха 4 | Train Loss: 0.8100 | Val ROC-AUC: 0.9317 | Val Profit: -1445
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9423
Precision: 0.274
Recall: 0.9524
F0.5-score: 0.3195
Profit: -1230

Эпоха 5 | Train Loss: 0.6128 | Val ROC-AUC: 0.9423 | Val Profit: -1230
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.9517
Precision: 0.2879
Recall: 0.9048
F0.5-score: 0.3333
Profit: -1090

Эпоха 6 | Train Loss: 0.4255 | Val ROC-AUC: 0.9517 | Val Profit: -1090
model_5_tolko_dropout | valid epoch 7
ROC-AUC: 0.956
Precision: 0.3167
Recall: 0.9048
F0.5-score: 0.364
Profit: -940

Эпоха 7 | Train Loss: 0.3588 | Val ROC-AUC: 0.9560 | Val Profit: -940
model_5_tolko_dropout | valid epoch 8
ROC-AUC: 0.9568
Precision: 0.3725
Recall: 0.9048
F0.5-score: 0.4222
Profit: -715

Эпоха 8 | Train Loss: 0.3134 | Val ROC-AUC: 0.9568

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.952
Precision: 0.4286
Recall: 0.8571
F0.5-score: 0.4762
Profit: -525

Эпоха 4 | Train Loss: 1.0006 | Val ROC-AUC: 0.9520 | Val Profit: -525
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9504
Precision: 0.3617
Recall: 0.8095
F0.5-score: 0.4067
Profit: -685

Эпоха 5 | Train Loss: 0.8070 | Val ROC-AUC: 0.9504 | Val Profit: -685
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.9586
Precision: 0.3778
Recall: 0.8095
F0.5-score: 0.4229
Profit: -635

Эпоха 6 | Train Loss: 0.5830 | Val ROC-AUC: 0.9586 | Val Profit: -635
model_5_tolko_dropout | valid epoch 7
ROC-AUC: 0.963
Precision: 0.3519
Recall: 0.9048
F0.5-score: 0.4008
Profit: -790

Эпоха 7 | Train Loss: 0.4533 | Val ROC-AUC: 0.9630 | Val Profit: -790
model_5_tolko_dropout | valid epoch 8
ROC-AUC: 0.9604
Precision: 0.3387
Recall: 1.0
F0.5-score: 0.3903
Profit: -920

Эпоха 8 | Train Loss: 0.4002 | Val ROC-AUC: 0.9604 | Val Profit: -920
model_5_tolko_dropout | valid epoch 9
ROC-AUC: 0.961
Pr

In [52]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.01, 'dropout': 0.025, 'profit': np.int64(-220), 'roc_auc': 0.98859825620389}


In [53]:
LR = best["lr"]
DROPOUT_COEF = best["dropout"]

In [54]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_5 = nn.Sequential(
    nn.Linear(9, 128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_5.parameters(), lr=LR)

config = {
    "model": "MLP_dropout_only",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_5)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_5_dropout_only", config=config)
log_file = new_log_file()

model_5 = train_model(
    model=model_5,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_5_tolko_dropout"
)

save_results(model_5, "model_5", log_file, run)

run.finish()


model_5_tolko_dropout | valid epoch 1
ROC-AUC: 0.9297
Precision: 0.2059
Recall: 1.0
F0.5-score: 0.2448
Profit: -1920

Эпоха 1 | Train Loss: 1.0901 | Val ROC-AUC: 0.9297 | Val Profit: -1920
model_5_tolko_dropout | valid epoch 2
ROC-AUC: 0.9222
Precision: 0.1721
Recall: 1.0
F0.5-score: 0.2063
Profit: -2420

Эпоха 2 | Train Loss: 0.6973 | Val ROC-AUC: 0.9222 | Val Profit: -2420
model_5_tolko_dropout | valid epoch 3
ROC-AUC: 0.9693
Precision: 0.2625
Recall: 1.0
F0.5-score: 0.3079
Profit: -1370

Эпоха 3 | Train Loss: 0.4525 | Val ROC-AUC: 0.9693 | Val Profit: -1370
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9272
Precision: 0.198
Recall: 0.9524
F0.5-score: 0.2353
Profit: -1930

Эпоха 4 | Train Loss: 0.6411 | Val ROC-AUC: 0.9272 | Val Profit: -1930
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9639
Precision: 0.2727
Recall: 1.0
F0.5-score: 0.3191
Profit: -1295

Эпоха 5 | Train Loss: 0.4349 | Val ROC-AUC: 0.9639 | Val Profit: -1295
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.9755

IOStream.flush timed out


model_5_tolko_dropout/train_loss,█▅▄▅▃▃▃▂▂▃▃▂▃▃▂▂▂▂▂▁▂▂▂▂▂▂▄▄▃▂▂▂▂▁▃▂▁▁▄▄
model_5_tolko_dropout/valid_f05,▂▁▃▂▃▃▄▄▅▂▃▃▂▃▃▄▅▄▅▅▄▅▅▃▄▄▁▂▄▅▇▆▅▇▃▅▇█▁▁
model_5_tolko_dropout/valid_precision,▂▁▃▂▃▃▄▄▅▂▃▃▂▃▃▄▅▄▅▅▃▅▄▃▄▄▁▂▄▅▇▆▅▇▃▅▇█▁▁
model_5_tolko_dropout/valid_profit,▃▂▅▃▅▅▆▆▇▄▆▅▄▅▅▆▇▆▇▇▆▇▇▆▆▆▂▄▆▇██▇█▆▇██▁▂
model_5_tolko_dropout/valid_recall,███▆██▆▆▁█████████▃██▃▆██████▆▆▃█▁▆████▆
model_5_tolko_dropout/valid_roc_auc,▂▁▆▂▅▇▆▇▆▄▇▇▆▇▇▇▆▇▇▇▇▇▅▅▅▆▂▅▇▇█▇▇█▅███▃▂
model_5_tolko_dropout/train_loss,0.58861
model_5_tolko_dropout/valid_f05,0.20121
model_5_tolko_dropout/valid_precision,0.16807
model_5_tolko_dropout/valid_profit,-2380
model_5_tolko_dropout/valid_recall,0.95238


In [55]:
train_metrics = evaluate_model(model_5, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_5, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9947
Precision: 0.6814
Recall: 0.9277
F0.5-score: 0.7196
Profit: -545

Test
ROC-AUC: 0.9694
Precision: 0.5621
Recall: 0.8474
F0.5-score: 0.6027
Profit: -309100



В этом эксперименте добавили только dropout и подбирали его коэффициент перебором. Лучшее качество получилось при dropout = 0.015

Увеличение колво эпох не улучшило ситуацию с переобучением

Это хуже бейзлайна (особенно если смотреть на Profit), но батчнорм показал себя лучше, чем дропаут в этом эксперименте

## model_6_batchnorm_i_dropout

In [56]:
EPOCHS = 30
SEED = 42

In [57]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005, 0.001]:
    for DROPOUT_COEF in [0.05, 0.1, 0.015, 0.2, 0.025, 0.3]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_6 = nn.Sequential(
            nn.Linear(9, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(16, 8),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(8, 1)
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_6.parameters(), lr=LR)
        model_6 = train_model(
            model=model_6,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_6_batchnorm_i_dropout"
        )

        val = evaluate_model(model_6, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "dropout": DROPOUT_COEF, "profit": val["profit"], "roc_auc": val["roc_auc"]})


model_6_batchnorm_i_dropout | valid epoch 1
ROC-AUC: 0.9645
Precision: 0.128
Recall: 1.0
F0.5-score: 0.1551
Profit: -3470

Эпоха 1 | Train Loss: 1.0060 | Val ROC-AUC: 0.9645 | Val Profit: -3470
model_6_batchnorm_i_dropout | valid epoch 2
ROC-AUC: 0.9569
Precision: 0.2234
Recall: 1.0
F0.5-score: 0.2645
Profit: -1720

Эпоха 2 | Train Loss: 0.5785 | Val ROC-AUC: 0.9569 | Val Profit: -1720
model_6_batchnorm_i_dropout | valid epoch 3
ROC-AUC: 0.9615
Precision: 0.2778
Recall: 0.9524
F0.5-score: 0.3236
Profit: -1205

Эпоха 3 | Train Loss: 0.4058 | Val ROC-AUC: 0.9615 | Val Profit: -1205
model_6_batchnorm_i_dropout | valid epoch 4
ROC-AUC: 0.9579
Precision: 0.3043
Recall: 1.0
F0.5-score: 0.3535
Profit: -1095

Эпоха 4 | Train Loss: 0.3390 | Val ROC-AUC: 0.9579 | Val Profit: -1095
model_6_batchnorm_i_dropout | valid epoch 5
ROC-AUC: 0.925
Precision: 0.2353
Recall: 0.9524
F0.5-score: 0.277
Profit: -1530

Эпоха 5 | Train Loss: 0.3588 | Val ROC-AUC: 0.9250 | Val Profit: -1530
model_6_batchnorm_i_dr

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [58]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.005, 'dropout': 0.05, 'profit': np.int64(-80), 'roc_auc': 0.979476861167002}


In [59]:
LR = best["lr"]
DROPOUT_COEF = best["dropout"]

In [60]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_6 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_6.parameters(), lr=LR)

config = {
    "model": "MLP_batchnorm_dropout",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_6)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_6_batchnorm_dropout", config=config)
log_file = new_log_file()

model_6 = train_model(
    model=model_6,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_6_batchnorm_i_dropout"
)


save_results(model_6, "model_6", log_file, run)

run.finish()

model_6_batchnorm_i_dropout | valid epoch 1
ROC-AUC: 0.9485
Precision: 0.14
Recall: 1.0
F0.5-score: 0.1691
Profit: -3120

Эпоха 1 | Train Loss: 1.1734 | Val ROC-AUC: 0.9485 | Val Profit: -3120
model_6_batchnorm_i_dropout | valid epoch 2
ROC-AUC: 0.9661
Precision: 0.1707
Recall: 1.0
F0.5-score: 0.2047
Profit: -2445

Эпоха 2 | Train Loss: 0.8037 | Val ROC-AUC: 0.9661 | Val Profit: -2445
model_6_batchnorm_i_dropout | valid epoch 3
ROC-AUC: 0.9725
Precision: 0.3387
Recall: 1.0
F0.5-score: 0.3903
Profit: -920

Эпоха 3 | Train Loss: 0.5405 | Val ROC-AUC: 0.9725 | Val Profit: -920
model_6_batchnorm_i_dropout | valid epoch 4
ROC-AUC: 0.9614
Precision: 0.3043
Recall: 1.0
F0.5-score: 0.3535
Profit: -1095

Эпоха 4 | Train Loss: 0.4330 | Val ROC-AUC: 0.9614 | Val Profit: -1095
model_6_batchnorm_i_dropout | valid epoch 5
ROC-AUC: 0.9696
Precision: 0.3387
Recall: 1.0
F0.5-score: 0.3903
Profit: -920

Эпоха 5 | Train Loss: 0.3722 | Val ROC-AUC: 0.9696 | Val Profit: -920
model_6_batchnorm_i_dropout | v

model_6_batchnorm_i_dropout/train_loss,█▆▄▄▃▃▂▂▂▂▂▂▂▂▃▂▂▁▁▁▂▂▂▂▁▁▁▁▁▁
model_6_batchnorm_i_dropout/valid_f05,▁▁▄▃▄▄▅▅▄▆▅▇▇▄▃▆▇▇█▇▄▄▆▇▇▅▆▇▇▆
model_6_batchnorm_i_dropout/valid_precision,▁▁▃▃▃▃▄▄▄▆▅▆▆▄▃▅▆▆█▇▃▃▅▆▇▆▇█▇▇
model_6_batchnorm_i_dropout/valid_profit,▁▃▆▆▆▆▇▇▇█▇██▇▆▇████▆▆▇███████
model_6_batchnorm_i_dropout/valid_recall,██████▇█▇▅█▅▆▇█▆▅▄▄▂█▇█▄▄▁▂▂▂▁
model_6_batchnorm_i_dropout/valid_roc_auc,▆▇█▇▇▇▇█▇██▇▆█▇████▁█▇███▇████
model_6_batchnorm_i_dropout/train_loss,0.02158
model_6_batchnorm_i_dropout/valid_f05,0.61538
model_6_batchnorm_i_dropout/valid_precision,0.72727
model_6_batchnorm_i_dropout/valid_profit,-100
model_6_batchnorm_i_dropout/valid_recall,0.38095


In [61]:
train_metrics = evaluate_model(model_6, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_6, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9951
Precision: 0.8913
Recall: 0.494
F0.5-score: 0.7678
Profit: -130

Test
ROC-AUC: 0.9532
Precision: 0.7722
Recall: 0.3921
F0.5-score: 0.6468
Profit: -94195



Если соединить 2 и дропаут и батчнорм вместе, то качество получается хорошим. Но оно чуть хуже, чем только при использовании batchnorm

## model_7_leaky_relu

In [62]:
EPOCHS = 30
SEED = 42

In [63]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005, 0.001]:
    for DROPOUT_COEF in [0, 0.05, 0.01, 0.015, 0.2, 0.025, 0.3]:
        for NEG_SLOPE in [0.01, 0.03, 0.05, 0.1]:
            random.seed(SEED)
            np.random.seed(SEED)
            torch.manual_seed(SEED) 

            model_7 = nn.Sequential(
                nn.Linear(9, 128),
                nn.BatchNorm1d(128),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(128, 64),
                nn.BatchNorm1d(64),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(64, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(8, 1)
            )

            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = torch.optim.Adam(model_7.parameters(), lr=LR)
            model_7 = train_model(
                model=model_7,
                train_loader=train_loader,
                X_valid=X_val,
                y_valid=y_val,
                loss_fn=loss_fn,
                optimizer=optimizer,
                epochs=EPOCHS,
                threshold=0.5,
                model_name="model_7_leaky_relu"
            )

            val = evaluate_model(model_7, X_val, y_val, threshold=0.5, name="val")
            results.append({"lr": LR, "dropout": DROPOUT_COEF, "neg_slope": NEG_SLOPE, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_7_leaky_relu | valid epoch 1
ROC-AUC: 0.9471
Precision: 0.1313
Recall: 1.0
F0.5-score: 0.1589
Profit: -3370

Эпоха 1 | Train Loss: 1.0485 | Val ROC-AUC: 0.9471 | Val Profit: -3370
model_7_leaky_relu | valid epoch 2
ROC-AUC: 0.9533
Precision: 0.2019
Recall: 1.0
F0.5-score: 0.2403
Profit: -1970

Эпоха 2 | Train Loss: 0.6157 | Val ROC-AUC: 0.9533 | Val Profit: -1970
model_7_leaky_relu | valid epoch 3
ROC-AUC: 0.9556
Precision: 0.2941
Recall: 0.9524
F0.5-score: 0.3413
Profit: -1105

Эпоха 3 | Train Loss: 0.4191 | Val ROC-AUC: 0.9556 | Val Profit: -1105
model_7_leaky_relu | valid epoch 4
ROC-AUC: 0.9697
Precision: 0.25
Recall: 1.0
F0.5-score: 0.2941
Profit: -1470

Эпоха 4 | Train Loss: 0.3685 | Val ROC-AUC: 0.9697 | Val Profit: -1470
model_7_leaky_relu | valid epoch 5
ROC-AUC: 0.9691
Precision: 0.5862
Recall: 0.8095
F0.5-score: 0.6204
Profit: -235

Эпоха 5 | Train Loss: 0.2854 | Val ROC-AUC: 0.9691 | Val Profit: -235
model_7_leaky_relu | valid epoch 6
ROC-AUC: 0.9751
Precision: 0.45
R

In [64]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.01, 'dropout': 0.025, 'neg_slope': 0.03, 'profit': np.int64(-15), 'roc_auc': 0.9833668678739101}


In [65]:
LR = best["lr"]
DROPOUT_COEF = best["dropout"]
NEG_SLOPE = best["neg_slope"]

In [66]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_7 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_7.parameters(), lr=LR)

config = {
    "model": "MLP_leaky_relu",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "negative_slope": NEG_SLOPE,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_7)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_7_leaky_relu", config=config)
log_file = new_log_file()

model_7 = train_model(
    model=model_7,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_7_leaky_relu"
)
save_results(model_7, "model_7", log_file, run)

run.finish()

model_7_leaky_relu | valid epoch 1
ROC-AUC: 0.9387
Precision: 0.1304
Recall: 1.0
F0.5-score: 0.1579
Profit: -3395

Эпоха 1 | Train Loss: 1.1545 | Val ROC-AUC: 0.9387 | Val Profit: -3395
model_7_leaky_relu | valid epoch 2
ROC-AUC: 0.9596
Precision: 0.1641
Recall: 1.0
F0.5-score: 0.197
Profit: -2570

Эпоха 2 | Train Loss: 0.7037 | Val ROC-AUC: 0.9596 | Val Profit: -2570
model_7_leaky_relu | valid epoch 3
ROC-AUC: 0.9639
Precision: 0.2333
Recall: 1.0
F0.5-score: 0.2756
Profit: -1620

Эпоха 3 | Train Loss: 0.4669 | Val ROC-AUC: 0.9639 | Val Profit: -1620
model_7_leaky_relu | valid epoch 4
ROC-AUC: 0.9675
Precision: 0.2838
Recall: 1.0
F0.5-score: 0.3312
Profit: -1220

Эпоха 4 | Train Loss: 0.4236 | Val ROC-AUC: 0.9675 | Val Profit: -1220
model_7_leaky_relu | valid epoch 5
ROC-AUC: 0.9531
Precision: 0.2561
Recall: 1.0
F0.5-score: 0.3009
Profit: -1420

Эпоха 5 | Train Loss: 0.3713 | Val ROC-AUC: 0.9531 | Val Profit: -1420
model_7_leaky_relu | valid epoch 6
ROC-AUC: 0.9679
Precision: 0.3846
Re

model_7_leaky_relu/train_loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▃▂▂▂▁▁▁▁▂▂▂▁▁▁▁
model_7_leaky_relu/valid_f05,▁▁▂▃▃▄▅▅▆▃▄▅▅█▂▃▅▅▇▇▅▁▂▃▄▆▇▆▄▆
model_7_leaky_relu/valid_precision,▁▁▂▂▂▃▅▄▆▃▄▄▅█▂▂▄▅▆▇▅▂▃▃▃▅▆▇▆▆
model_7_leaky_relu/valid_profit,▁▃▅▆▅▇▇▇█▆▇▇▇█▅▅▇▇█████▆▇▇████
model_7_leaky_relu/valid_recall,███████▅▄█▇█▆▅███▆▅▅▃▁▁██▇▅▄▂▄
model_7_leaky_relu/valid_roc_auc,▁▄▅▆▃▆▇▆▃▆▅▅▇█▅▅▆▆▆▆▆▄▄█▇▇▆▇▆▆
model_7_leaky_relu/train_loss,0.05906
model_7_leaky_relu/valid_f05,0.61728
model_7_leaky_relu/valid_precision,0.66667
model_7_leaky_relu/valid_profit,-130
model_7_leaky_relu/valid_recall,0.47619


In [67]:
train_metrics = evaluate_model(model_7, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_7, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.992
Precision: 0.8947
Recall: 0.6145
F0.5-score: 0.8199
Profit: -55

Test
ROC-AUC: 0.9765
Precision: 0.7567
Recall: 0.5004
F0.5-score: 0.6864
Profit: -95330



Поменяли функцию активации с ReLU на LeakyReLU и перебрали LR, DROPOUT_COEF и NEG_SLOPE. Получилось добиться лучшего качества по Profit в -95к

Да, это лучше бейзлайна, но качество ухудшилось. 

Вывод: особого эффекта LeakyRelu не дал, поэтому остановимся на обычном ReLu в будущем 

## model_8_weight_decay

In [68]:
EPOCHS = 40
SEED = 42

In [69]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005, 0.001]:
    for DROPOUT_COEF in [0, 0.05, 0.01, 0.015, 0.2, 0.025, 0.3]:
        for WEIGHT_DECAY in [0.0001, 0.001, 0.01]:
            random.seed(SEED)
            np.random.seed(SEED)
            torch.manual_seed(SEED) 

            model_8 = nn.Sequential(
                nn.Linear(9, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(128, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(64, 32),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(8, 1)
            )

            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = torch.optim.Adam(model_8.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            model_8 = train_model(
                model=model_8,
                train_loader=train_loader,
                X_valid=X_val,
                y_valid=y_val,
                loss_fn=loss_fn,
                optimizer=optimizer,
                epochs=EPOCHS,
                threshold=0.5,
                model_name="model_8_weight_decay"
            )

            val = evaluate_model(model_8, X_val, y_val, threshold=0.5, name="val")
            results.append({"lr": LR, "dropout": DROPOUT_COEF, "weight_decay": WEIGHT_DECAY, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_8_weight_decay | valid epoch 1
ROC-AUC: 0.952
Precision: 0.1235
Recall: 1.0
F0.5-score: 0.1498
Profit: -3620

Эпоха 1 | Train Loss: 1.0016 | Val ROC-AUC: 0.9520 | Val Profit: -3620
model_8_weight_decay | valid epoch 2
ROC-AUC: 0.9624
Precision: 0.2165
Recall: 1.0
F0.5-score: 0.2567
Profit: -1795

Эпоха 2 | Train Loss: 0.5536 | Val ROC-AUC: 0.9624 | Val Profit: -1795
model_8_weight_decay | valid epoch 3
ROC-AUC: 0.9635
Precision: 0.2414
Recall: 1.0
F0.5-score: 0.2846
Profit: -1545

Эпоха 3 | Train Loss: 0.4543 | Val ROC-AUC: 0.9635 | Val Profit: -1545
model_8_weight_decay | valid epoch 4
ROC-AUC: 0.9693
Precision: 0.4651
Recall: 0.9524
F0.5-score: 0.5181
Profit: -480

Эпоха 4 | Train Loss: 0.3237 | Val ROC-AUC: 0.9693 | Val Profit: -480
model_8_weight_decay | valid epoch 5
ROC-AUC: 0.9544
Precision: 0.253
Recall: 1.0
F0.5-score: 0.2975
Profit: -1445

Эпоха 5 | Train Loss: 0.3354 | Val ROC-AUC: 0.9544 | Val Profit: -1445
model_8_weight_decay | valid epoch 6
ROC-AUC: 0.974
Precision

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 9
ROC-AUC: 0.9726
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 9 | Train Loss: 0.2940 | Val ROC-AUC: 0.9726 | Val Profit: -105
model_8_weight_decay | valid epoch 10
ROC-AUC: 0.9746
Precision: 0.4318
Recall: 0.9048
F0.5-score: 0.4822
Profit: -540

Эпоха 10 | Train Loss: 0.2959 | Val ROC-AUC: 0.9746 | Val Profit: -540
model_8_weight_decay | valid epoch 11
ROC-AUC: 0.9693
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 11 | Train Loss: 0.2810 | Val ROC-AUC: 0.9693 | Val Profit: -105
model_8_weight_decay | valid epoch 12
ROC-AUC: 0.974
Precision: 0.6667
Recall: 0.2857
F0.5-score: 0.5263
Profit: -120

Эпоха 12 | Train Loss: 0.3010 | Val ROC-AUC: 0.9740 | Val Profit: -120
model_8_weight_decay | valid epoch 13
ROC-AUC: 0.973
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 13 | Train Loss: 0.2475 | Val ROC-AUC: 0.9730 | Val Profit: -105
model_8_weight_decay | valid epoch 14
ROC-AUC: 0.9819
Precision: 0.8333
Reca

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 12
ROC-AUC: 0.9784
Precision: 0.2692
Recall: 1.0
F0.5-score: 0.3153
Profit: -1320

Эпоха 12 | Train Loss: 0.3251 | Val ROC-AUC: 0.9784 | Val Profit: -1320
model_8_weight_decay | valid epoch 13
ROC-AUC: 0.97
Precision: 0.48
Recall: 0.5714
F0.5-score: 0.4959
Profit: -310

Эпоха 13 | Train Loss: 0.2819 | Val ROC-AUC: 0.9700 | Val Profit: -310
model_8_weight_decay | valid epoch 14
ROC-AUC: 0.9696
Precision: 0.5455
Recall: 0.2857
F0.5-score: 0.4615
Profit: -170

Эпоха 14 | Train Loss: 0.2534 | Val ROC-AUC: 0.9696 | Val Profit: -170
model_8_weight_decay | valid epoch 15
ROC-AUC: 0.9675
Precision: 0.5
Recall: 0.5714
F0.5-score: 0.5128
Profit: -285

Эпоха 15 | Train Loss: 0.2547 | Val ROC-AUC: 0.9675 | Val Profit: -285
model_8_weight_decay | valid epoch 16
ROC-AUC: 0.9691
Precision: 0.4828
Recall: 0.6667
F0.5-score: 0.5109
Profit: -340

Эпоха 16 | Train Loss: 0.3021 | Val ROC-AUC: 0.9691 | Val Profit: -340
model_8_weight_decay | valid epoch 17
ROC-AUC: 0.9763

...

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 24
ROC-AUC: 0.8184
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 24 | Train Loss: 0.1883 | Val ROC-AUC: 0.8184 | Val Profit: -105
model_8_weight_decay | valid epoch 25
ROC-AUC: 0.803
Precision: 0.4286
Recall: 0.1429
F0.5-score: 0.3061
Profit: -175

Эпоха 25 | Train Loss: 0.1870 | Val ROC-AUC: 0.8030 | Val Profit: -175
model_8_weight_decay | valid epoch 26
ROC-AUC: 0.8928
Precision: 0.375
Recall: 0.1429
F0.5-score: 0.283
Profit: -200

Эпоха 26 | Train Loss: 0.1932 | Val ROC-AUC: 0.8928 | Val Profit: -200
model_8_weight_decay | valid epoch 27
ROC-AUC: 0.8711
Precision: 0.4286
Recall: 0.1429
F0.5-score: 0.3061
Profit: -175

Эпоха 27 | Train Loss: 0.1719 | Val ROC-AUC: 0.8711 | Val Profit: -175
model_8_weight_decay | valid epoch 28
ROC-AUC: 0.9152
Precision: 0.6
Recall: 0.1429
F0.5-score: 0.3659
Profit: -125

Эпоха 28 | Train Loss: 0.1546 | Val ROC-AUC: 0.9152 | Val Profit: -125
model_8_weight_decay | valid epoch 29
ROC-AUC: 0.8744
Precis

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 37
ROC-AUC: 0.8913
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -155

Эпоха 37 | Train Loss: 0.0730 | Val ROC-AUC: 0.8913 | Val Profit: -155
model_8_weight_decay | valid epoch 38
ROC-AUC: 0.9231
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 38 | Train Loss: 0.0808 | Val ROC-AUC: 0.9231 | Val Profit: -105
model_8_weight_decay | valid epoch 39
ROC-AUC: 0.8452
Precision: 0.6
Recall: 0.1429
F0.5-score: 0.3659
Profit: -125

Эпоха 39 | Train Loss: 0.1025 | Val ROC-AUC: 0.8452 | Val Profit: -125
model_8_weight_decay | valid epoch 40
ROC-AUC: 0.8044
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 40 | Train Loss: 0.0975 | Val ROC-AUC: 0.8044 | Val Profit: -105

Лучшая эпоха для model_8_weight_decay: 34
Лучший ROC-AUC: 0.9415157612340711
val
ROC-AUC: 0.9415
Precision: 0.5
Recall: 0.0476
F0.5-score: 0.1724
Profit: -120



/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [70]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.01, 'dropout': 0.015, 'weight_decay': 0.01, 'profit': np.int64(-20), 'roc_auc': 0.9876592890677398}


In [71]:
LR = best["lr"]
WEIGHT_DECAY = best["weight_decay"]
DROPOUT_COEF = best["dropout"]

In [72]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_8 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_8.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "MLP_weight_decay",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_8)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_8_weight_decay", config=config)
log_file = new_log_file()

model_8 = train_model(
    model=model_8,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_8_weight_decay"
)

save_results(model_8, "model_8", log_file, run)

run.finish()

model_8_weight_decay | valid epoch 1
ROC-AUC: 0.9366
Precision: 0.1221
Recall: 1.0
F0.5-score: 0.1481
Profit: -3670

Эпоха 1 | Train Loss: 1.1236 | Val ROC-AUC: 0.9366 | Val Profit: -3670
model_8_weight_decay | valid epoch 2
ROC-AUC: 0.9626
Precision: 0.1927
Recall: 1.0
F0.5-score: 0.2298
Profit: -2095

Эпоха 2 | Train Loss: 0.6802 | Val ROC-AUC: 0.9626 | Val Profit: -2095
model_8_weight_decay | valid epoch 3
ROC-AUC: 0.9744
Precision: 0.4118
Recall: 1.0
F0.5-score: 0.4667
Profit: -645

Эпоха 3 | Train Loss: 0.4570 | Val ROC-AUC: 0.9744 | Val Profit: -645
model_8_weight_decay | valid epoch 4
ROC-AUC: 0.9564
Precision: 0.2763
Recall: 1.0
F0.5-score: 0.3231
Profit: -1270

Эпоха 4 | Train Loss: 0.4240 | Val ROC-AUC: 0.9564 | Val Profit: -1270
model_8_weight_decay | valid epoch 5
ROC-AUC: 0.9725
Precision: 0.3962
Recall: 1.0
F0.5-score: 0.4506
Profit: -695

Эпоха 5 | Train Loss: 0.4137 | Val ROC-AUC: 0.9725 | Val Profit: -695
model_8_weight_decay | valid epoch 6
ROC-AUC: 0.9714
Precision: 

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 14
ROC-AUC: 0.9705
Precision: 0.2877
Recall: 1.0
F0.5-score: 0.3355
Profit: -1195

Эпоха 14 | Train Loss: 0.2652 | Val ROC-AUC: 0.9705 | Val Profit: -1195
model_8_weight_decay | valid epoch 15
ROC-AUC: 0.9714
Precision: 0.5
Recall: 0.9048
F0.5-score: 0.5491
Profit: -390

Эпоха 15 | Train Loss: 0.2847 | Val ROC-AUC: 0.9714 | Val Profit: -390
model_8_weight_decay | valid epoch 16
ROC-AUC: 0.976
Precision: 0.5882
Recall: 0.4762
F0.5-score: 0.5618
Profit: -180

Эпоха 16 | Train Loss: 0.2388 | Val ROC-AUC: 0.9760 | Val Profit: -180
model_8_weight_decay | valid epoch 17
ROC-AUC: 0.9745
Precision: 0.5455
Recall: 0.2857
F0.5-score: 0.4615
Profit: -170

Эпоха 17 | Train Loss: 0.2398 | Val ROC-AUC: 0.9745 | Val Profit: -170
model_8_weight_decay | valid epoch 18
ROC-AUC: 0.9599
Precision: 0.5
Recall: 0.0952
F0.5-score: 0.2703
Profit: -135

Эпоха 18 | Train Loss: 0.2173 | Val ROC-AUC: 0.9599 | Val Profit: -135
model_8_weight_decay | valid epoch 19
ROC-AUC: 0.9755

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 28
ROC-AUC: 0.9706
Precision: 0.1981
Recall: 1.0
F0.5-score: 0.236
Profit: -2020

Эпоха 28 | Train Loss: 0.2670 | Val ROC-AUC: 0.9706 | Val Profit: -2020
model_8_weight_decay | valid epoch 29
ROC-AUC: 0.9831
Precision: 0.5135
Recall: 0.9048
F0.5-score: 0.5621
Profit: -365

Эпоха 29 | Train Loss: 0.2888 | Val ROC-AUC: 0.9831 | Val Profit: -365
model_8_weight_decay | valid epoch 30
ROC-AUC: 0.8001
Precision: 0.5455
Recall: 0.2857
F0.5-score: 0.4615
Profit: -170

Эпоха 30 | Train Loss: 0.2560 | Val ROC-AUC: 0.8001 | Val Profit: -170
model_8_weight_decay | valid epoch 31
ROC-AUC: 0.9818
Precision: 0.5625
Recall: 0.8571
F0.5-score: 0.604
Profit: -275

Эпоха 31 | Train Loss: 0.2562 | Val ROC-AUC: 0.9818 | Val Profit: -275
model_8_weight_decay | valid epoch 32
ROC-AUC: 0.9744
Precision: 0.3333
Recall: 0.0476
F0.5-score: 0.1515
Profit: -145

Эпоха 32 | Train Loss: 0.2288 | Val ROC-AUC: 0.9744 | Val Profit: -145
model_8_weight_decay | valid epoch 33
ROC-AUC: 0

model_8_weight_decay/train_loss,█▅▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▂▁▁▁▂▁▁▁▁▂▂▂▂▁▂▂▁▁▂▁▁▁
model_8_weight_decay/valid_f05,▂▃▅▄▅▆▂▄▅▁▆▅▇▄▆▆▅▃▅▄▅▅▃▆▅▅▁▃▆▅▆▂▅▁▁▅▄▅▄█
model_8_weight_decay/valid_precision,▂▃▄▃▄▅▅▃▅▁▅▅▆▃▅▆▆▅▄▅▅▄▂▅▅▅▁▃▅▆▆▄▄▁▁▅▄▄▅█
model_8_weight_decay/valid_profit,▁▄▇▆▇▇█▆▇██▇█▆▇███▆█▇▇▃▇█▇█▄▇███▇██▇▆▇██
model_8_weight_decay/valid_recall,█████▆▁██▁▆█▇█▇▄▃▂█▂▇██▇▃▄▁█▇▃▇▁█▁▁▄█▆▂▆
model_8_weight_decay/valid_roc_auc,▆▇█▇▇▇▇▇██▇▇█▇▇██▇██▇▇█▇▇▇▄▇█▁████▇▇▇▇▇█
model_8_weight_decay/train_loss,0.24065
model_8_weight_decay/valid_f05,0.82474
model_8_weight_decay/valid_precision,0.84211
model_8_weight_decay/valid_profit,-20
model_8_weight_decay/valid_recall,0.7619


In [73]:
train_metrics = evaluate_model(model_8, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_8, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.992
Precision: 0.8413
Recall: 0.6386
F0.5-score: 0.791
Profit: -135

Test
ROC-AUC: 0.9833
Precision: 0.7402
Recall: 0.5794
F0.5-score: 0.7013
Profit: -101765



Добавив регуляризацию видим, что качество улучшилось относительно бейзлайна, но все еще не стало лучшим показателем из всех моделей. Странно, но использование регуляризации проблему с сильным переобучением нейронной сети это не решила

## model_9_Focal_loss

In [81]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        targets = targets.float()

        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        loss = focal_weight * bce_loss

        return loss.mean()

In [82]:
EPOCHS = 30
SEED = 42

In [83]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005]:
    for GAMMA in [1.0, 2.0, 3.0]:
        for ALPHA in [0.25, 0.5, 0.99]:
            for DROPOUT_COEF in [0.0, 0.1, 0.2]:
                for WEIGHT_DECAY in [0.0, 0.0001, 0.001]:
                    random.seed(SEED)
                    np.random.seed(SEED)
                    torch.manual_seed(SEED) 

                    model_9 = nn.Sequential(
                        nn.Linear(9, 128),
                        nn.BatchNorm1d(128),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(128, 64),
                        nn.BatchNorm1d(64),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(64, 32),
                        nn.BatchNorm1d(32),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(32, 16),
                        nn.BatchNorm1d(16),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(16, 8),
                        nn.BatchNorm1d(8),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(8, 1)
                    )

                    loss_fn = FocalLoss(alpha=ALPHA, gamma=GAMMA)
                    optimizer = torch.optim.Adam(model_9.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
                    model_9 = train_model(
                        model=model_9,
                        train_loader=train_loader,
                        X_valid=X_val,
                        y_valid=y_val,
                        loss_fn=loss_fn,
                        optimizer=optimizer,
                        epochs=EPOCHS,
                        threshold=0.5,
                        model_name="model_9_Focal_loss"
                    )
                    val = evaluate_model(model_9, X_val, y_val, threshold=0.5, name="val")
                    results.append({"lr": LR, "gamma": GAMMA, "alpha": ALPHA, "dropout": DROPOUT_COEF, "weight_decay": WEIGHT_DECAY, "profit": val["profit"], "roc_auc": val["roc_auc"]})

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 1
ROC-AUC: 0.8952
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 1 | Train Loss: 0.0426 | Val ROC-AUC: 0.8952 | Val Profit: -105
model_9_Focal_loss | valid epoch 2
ROC-AUC: 0.9559
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 2 | Train Loss: 0.0189 | Val ROC-AUC: 0.9559 | Val Profit: -105
model_9_Focal_loss | valid epoch 3
ROC-AUC: 0.974
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 3 | Train Loss: 0.0153 | Val ROC-AUC: 0.9740 | Val Profit: -105
model_9_Focal_loss | valid epoch 4
ROC-AUC: 0.96
Precision: 0.5294
Recall: 0.4286
F0.5-score: 0.5056
Profit: -215

Эпоха 4 | Train Loss: 0.0126 | Val ROC-AUC: 0.9600 | Val Profit: -215
model_9_Focal_loss | valid epoch 5
ROC-AUC: 0.9502
Precision: 0.4138
Recall: 0.5714
F0.5-score: 0.438
Profit: -410

Эпоха 5 | Train Loss: 0.0110 | Val ROC-AUC: 0.9502 | Val Profit: -410
model_9_Focal_loss | valid epoch 6
ROC-AUC: 0.9547
Precision: 0.4828
Recall: 0.6667
F0.5-score: 

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_9_Focal_loss | valid epoch 6
ROC-AUC: 0.9731
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 6 | Train Loss: 0.0124 | Val ROC-AUC: 0.9731 | Val Profit: -105
model_9_Focal_loss | valid epoch 7
ROC-AUC: 0.9661
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 7 | Train Loss: 0.0128 | Val ROC-AUC: 0.9661 | Val Profit: -105
model_9_Focal_loss | valid epoch 8
ROC-AUC: 0.9655
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 8 | Train Loss: 0.0119 | Val ROC-AUC: 0.9655 | Val Profit: -105
model_9_Focal_loss | valid epoch 9
ROC-AUC: 0.9638
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 9 | Train Loss: 0.0131 | Val ROC-AUC: 0.9638 | Val Profit: -105
model_9_Focal_loss | valid epoch 10
ROC-AUC: 0.9502
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 10 | Train Loss: 0.0120 | Val ROC-AUC: 0.9502 | Val Profit: -105
model_9_Focal_loss | valid epoch 11
ROC-AUC: 0.9753
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105


/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

...

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_9_Focal_loss | valid epoch 10
ROC-AUC: 0.9019
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -130

Эпоха 10 | Train Loss: 0.0116 | Val ROC-AUC: 0.9019 | Val Profit: -130
model_9_Focal_loss | valid epoch 11
ROC-AUC: 0.9623
Precision: 0.4242
Recall: 0.6667
F0.5-score: 0.4575
Profit: -440

Эпоха 11 | Train Loss: 0.0119 | Val ROC-AUC: 0.9623 | Val Profit: -440
model_9_Focal_loss | valid epoch 12
ROC-AUC: 0.972
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 12 | Train Loss: 0.0124 | Val ROC-AUC: 0.9720 | Val Profit: -105
model_9_Focal_loss | valid epoch 13
ROC-AUC: 0.974
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 13 | Train Loss: 0.0101 | Val ROC-AUC: 0.9740 | Val Profit: -105
model_9_Focal_loss | valid epoch 14
ROC-AUC: 0.9313
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 14 | Train Loss: 0.0074 | Val ROC-AUC: 0.9313 | Val Profit: -105
model_9_Focal_loss | valid epoch 15
ROC-AUC: 0.971
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 17
ROC-AUC: 0.9677
Precision: 0.6364
Recall: 0.3333
F0.5-score: 0.5385
Profit: -135

Эпоха 17 | Train Loss: 0.0105 | Val ROC-AUC: 0.9677 | Val Profit: -135
model_9_Focal_loss | valid epoch 18
ROC-AUC: 0.9761
Precision: 1.0
Recall: 0.1429
F0.5-score: 0.4545
Profit: -75

Эпоха 18 | Train Loss: 0.0079 | Val ROC-AUC: 0.9761 | Val Profit: -75
model_9_Focal_loss | valid epoch 19
ROC-AUC: 0.9611
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 19 | Train Loss: 0.0072 | Val ROC-AUC: 0.9611 | Val Profit: -105
model_9_Focal_loss | valid epoch 20
ROC-AUC: 0.9556
Precision: 0.5
Recall: 0.2381
F0.5-score: 0.4098
Profit: -180

Эпоха 20 | Train Loss: 0.0063 | Val ROC-AUC: 0.9556 | Val Profit: -180
model_9_Focal_loss | valid epoch 21
ROC-AUC: 0.9728
Precision: 1.0
Recall: 0.0476
F0.5-score: 0.2
Profit: -95

Эпоха 21 | Train Loss: 0.0057 | Val ROC-AUC: 0.9728 | Val Profit: -95
model_9_Focal_loss | valid epoch 22
ROC-AUC: 0.9757
Precision: 0.5455
Recall: 0.

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [84]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.01, 'gamma': 2.0, 'alpha': 0.25, 'dropout': 0.0, 'weight_decay': 0.001, 'profit': np.int64(-50), 'roc_auc': 0.9904761904761905}


In [85]:
LR = best["lr"]
GAMMA = best["gamma"]
ALPHA = best["alpha"]
DROPOUT_COEF = best["dropout"]
WEIGHT_DECAY = best["weight_decay"]

In [86]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_9 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = FocalLoss(alpha=ALPHA, gamma=GAMMA)
optimizer = torch.optim.Adam(model_9.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "MLP_focal_loss",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "alpha": ALPHA,
    "gamma": GAMMA,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_9)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_9_focal_loss", config=config)
log_file = new_log_file()

model_9 = train_model(
    model=model_9,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_9_Focal_loss"
)


save_results(model_9, "model_9", log_file, run)

run.finish()

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_9_Focal_loss | valid epoch 1
ROC-AUC: 0.9359
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 1 | Train Loss: 0.0181 | Val ROC-AUC: 0.9359 | Val Profit: -105
model_9_Focal_loss | valid epoch 2
ROC-AUC: 0.9399
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 2 | Train Loss: 0.0087 | Val ROC-AUC: 0.9399 | Val Profit: -105
model_9_Focal_loss | valid epoch 3
ROC-AUC: 0.9545
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 3 | Train Loss: 0.0079 | Val ROC-AUC: 0.9545 | Val Profit: -105
model_9_Focal_loss | valid epoch 4
ROC-AUC: 0.9584
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 4 | Train Loss: 0.0076 | Val ROC-AUC: 0.9584 | Val Profit: -105
model_9_Focal_loss | valid epoch 5
ROC-AUC: 0.972
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 5 | Train Loss: 0.0069 | Val ROC-AUC: 0.9720 | Val Profit: -105
model_9_Focal_loss | valid epoch 6
ROC-AUC: 0.9746
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпо

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

model_9_Focal_loss | valid epoch 13
ROC-AUC: 0.9842
Precision: 0.3962
Recall: 1.0
F0.5-score: 0.4506
Profit: -695

Эпоха 13 | Train Loss: 0.0059 | Val ROC-AUC: 0.9842 | Val Profit: -695
model_9_Focal_loss | valid epoch 14
ROC-AUC: 0.9812
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 14 | Train Loss: 0.0061 | Val ROC-AUC: 0.9812 | Val Profit: -105
model_9_Focal_loss | valid epoch 15
ROC-AUC: 0.9795
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 15 | Train Loss: 0.0059 | Val ROC-AUC: 0.9795 | Val Profit: -105
model_9_Focal_loss | valid epoch 16
ROC-AUC: 0.9866
Precision: 1.0
Recall: 0.0476
F0.5-score: 0.2
Profit: -95

Эпоха 16 | Train Loss: 0.0056 | Val ROC-AUC: 0.9866 | Val Profit: -95
model_9_Focal_loss | valid epoch 17
ROC-AUC: 0.9889
Precision: 0.3962
Recall: 1.0
F0.5-score: 0.4506
Profit: -695

Эпоха 17 | Train Loss: 0.0056 | Val ROC-AUC: 0.9889 | Val Profit: -695
model_9_Focal_loss | valid epoch 18
ROC-AUC: 0.9851
Precision: 0.0
Recall: 0.0
F0.5-sco

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/max

...

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 25
ROC-AUC: 0.9894
Precision: 1.0
Recall: 0.0476
F0.5-score: 0.2
Profit: -95

Эпоха 25 | Train Loss: 0.0051 | Val ROC-AUC: 0.9894 | Val Profit: -95
model_9_Focal_loss | valid epoch 26
ROC-AUC: 0.983
Precision: 1.0
Recall: 0.2381
F0.5-score: 0.6098
Profit: -55

Эпоха 26 | Train Loss: 0.0053 | Val ROC-AUC: 0.9830 | Val Profit: -55
model_9_Focal_loss | valid epoch 27
ROC-AUC: 0.9847
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 27 | Train Loss: 0.0052 | Val ROC-AUC: 0.9847 | Val Profit: -105
model_9_Focal_loss | valid epoch 28
ROC-AUC: 0.9905
Precision: 0.8889
Recall: 0.381
F0.5-score: 0.7018
Profit: -50

Эпоха 28 | Train Loss: 0.0052 | Val ROC-AUC: 0.9905 | Val Profit: -50
model_9_Focal_loss | valid epoch 29
ROC-AUC: 0.9898
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 29 | Train Loss: 0.0053 | Val ROC-AUC: 0.9898 | Val Profit: -105
model_9_Focal_loss | valid epoch 30
ROC-AUC: 0.9899
Precision: 0.9167
Recall: 0.5238
F0.5-

/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxsafonov/gp-5/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Test
ROC-AUC: 0.9823
Precision: 0.8896
Recall: 0.3607
F0.5-score: 0.6879
Profit: -59605



model_9_Focal_loss/train_loss,█▃▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁
model_9_Focal_loss/valid_f05,▁▁▁▁▁▁▆▁▁▁▁▁▅▁▁▃▅▁▁▁▄▄▁▁▃▆▁▇▁█
model_9_Focal_loss/valid_precision,▁▁▁▁▁▁▆▁▁▁▁▁▄▁▁█▄▁▁▁▃█▁▁██▁▇▁▇
model_9_Focal_loss/valid_profit,██████▇█████▄███▄███▁█████████
model_9_Focal_loss/valid_recall,▁▁▁▁▁▁▅▁▁▁▁▁█▁▁▁█▁▁▁█▂▁▁▁▃▁▄▁▅
model_9_Focal_loss/valid_roc_auc,▁▂▃▄▆▆▇▆▇▇▇▆▇▇▇██▇▇█▇▇███▇▇███
model_9_Focal_loss/train_loss,0.0054
model_9_Focal_loss/valid_f05,0.7971
model_9_Focal_loss/valid_precision,0.91667
model_9_Focal_loss/valid_profit,-20
model_9_Focal_loss/valid_recall,0.52381


In [87]:
train_metrics = evaluate_model(model_9, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_9, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9882
Precision: 0.9412
Recall: 0.3855
F0.5-score: 0.7306
Profit: -145

Test
ROC-AUC: 0.9823
Precision: 0.8896
Recall: 0.3607
F0.5-score: 0.6879
Profit: -59605



В этом эксперименте пробовали ментять функцию активации и это очень сильно помогло! Profit поднялся до -59к. Использовал FocalLoss. И перебирали параметры

Но все еще проблема с переобучением не решилась



В этом эксперименте пробовали ментять функцию потеь, но это особо не помогло. Использовал FocalLoss

Параметр альфа = вес класса 1. Поставил его побольше, так как у нас 1 в 17 раз меньше чем 0. Нам важны ошибки, которые модель допускает на 1.

Параметр гамма = Насколько быстро модель забывает "легкие" нули и фокусируется на сложных границах между классами. Обычно ставят по умолчанию 2.

В случае FocalLoss мы мало штрафуем за уверенные ответы и сильно штрафуем за неуверенные. Отлично подходит для задач, когда сильный дизбаланс классов.

Но в нашем случае это не дало сильного прироста эффекта. Получилось улучшить на 2к всего от нашего лучшего предыдущего эффекта. Но тем не менее эта архитектура наиболее эффективной получилась сейчас

# Ансамбль из лучших моделей

In [91]:
EPOCHS = 120
LR = 0.001
WEIGHT_DECAY = 1e-4
N_ENSEMBLE = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_9 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = FocalLoss(alpha=ALPHA, gamma=GAMMA)
optimizer = torch.optim.Adam(model_9.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "Ensemble",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "n_ensemble": N_ENSEMBLE,
    "architecture": str(model_9)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="ensemble", config=config)

In [93]:
def build_model():
    return nn.Sequential(
        nn.Linear(9, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(DROPOUT_COEF),

        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(DROPOUT_COEF),

        nn.Linear(64, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(DROPOUT_COEF),

        nn.Linear(32, 16),
        nn.BatchNorm1d(16),
        nn.ReLU(),
        nn.Dropout(DROPOUT_COEF),

        nn.Linear(16, 8),
        nn.BatchNorm1d(8),
        nn.ReLU(),
        nn.Dropout(DROPOUT_COEF),

        nn.Linear(8, 1)
    )

test_probs = []
logging.info("Запустили обучение моделей ансамбля")

np.random.seed(42)
seeds = np.random.randint(1, 525252, size=N_ENSEMBLE)
ensemble = []
for i in range(N_ENSEMBLE):
    seed = seeds[i]
    torch.manual_seed(seed)
    net = build_model()
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    logging.info(f"Запустили обучение модели {i+1}. Сид: {seed}")


    net.train()
    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            epoch_loss += loss.item()
            opt.step()
        
        logging.info(f"эпоха: {epoch}, loss:  {round(epoch_loss, 4)}")
        wandb.log({"epoch": epoch, f"ensemble/{i}/train_loss": epoch_loss})
    
    ensemble.append(net.state_dict())

    net.eval()
    with torch.no_grad():
        test_probs.append(torch.sigmoid(net(X_test)).numpy())

    logging.info(f"Сеть {i} обучена")
    print("сеть", i, "обучена")

probs2 = np.mean(test_probs, axis=0)

сеть 0 обучена
сеть 1 обучена
сеть 2 обучена
сеть 3 обучена
сеть 4 обучена


In [115]:
y_pred_2 = (probs2 >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_2).ravel()
test_profit = tp * 5 - fp * 25 - fn * 5
test_profit

np.int64(-60675)

Применив ансамбль из 5 лучших моделей, видим, что качество немного скорректировалось с -57к до -60к. 

In [96]:
auc_score = roc_auc_score(y_test, probs2)
logging.info(f"На тесте ROC_AUC: {auc_score}")
auc_score

0.9760348268814318

In [97]:
pre_score = average_precision_score(y_test, probs2)
logging.info(f"На тесте Precision: {pre_score}")
pre_score

0.7593236362962668

In [104]:
results = wandb.Table(columns=['model', 'test/avg_roc_auc', 'test/avg_precision', 'profit'])
results.add_data("ensemble", auc_score, pre_score, test_profit)
wandb.log({"results": results})

In [105]:
import pickle

pickle.dump(ensemble, open("models/model_fraud_ensemble.pkl", 'wb'))
logging.info("Сохранили веса моделей ансамбля в папку models")

In [106]:
artifact = wandb.Artifact(name="model_fraud_ensemble.pkl", type="model", description="Ансамбль моделей для определения Фрода")

artifact.add_file("models/model_fraud_ensemble.pkl")
artifact.add_file(log_file)
run.log_artifact(artifact)


<Artifact model_fraud_ensemble.pkl>

In [107]:
run.finish()
wandb.finish()

ensemble/0/train_loss,█▅▅▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
ensemble/1/train_loss,█▇▅▄▄▃▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
ensemble/2/train_loss,█▄▃▃▃▂▂▂▂▂▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
ensemble/3/train_loss,█▇▅▅▃▃▃▃▂▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▃▂▂▂▁▁▁▁▁▁▁▁
ensemble/4/train_loss,█▆▃▃▂▂▂▂▂▁▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▂▁▁▂▂▂▂▁▁
epoch,▂▃▃▄▅▆█▁▁▂▄▅▅▇▃▃▄▅▆▇▁▂▃▄▄▄▅▆▁▁▂▂▃▃▄▅▅▆▆█
ensemble/0/train_loss,0.06894
ensemble/1/train_loss,0.04355
ensemble/2/train_loss,0.04368
ensemble/3/train_loss,0.04809
ensemble/4/train_loss,0.0347


Ради интереса посмотрим на knn

In [112]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

k_values = [3, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]

best_k = None
best_roc_auc = -1
best_knn = None
X_train_np = X_train.numpy()
y_train_np = y_train.numpy().ravel()
X_val_np = X_val.numpy()
y_val_np = y_val.numpy().ravel()
X_test_np = X_test.numpy()
y_test_np = y_test.numpy().ravel()

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights="distance")

    knn.fit(X_train_np, y_train_np)
    val_probs = knn.predict_proba(X_val_np)[:, 1]
    val_roc_auc = roc_auc_score(y_val_np, val_probs)

    print(f"k={k} получаем val ROC-AUC={val_roc_auc:.4f}")

    if val_roc_auc > best_roc_auc:
        best_roc_auc = val_roc_auc
        best_k = k
        best_knn = knn

print("\nЛучший KNN")
print("best k:", best_k)
print("best val ROC-AUC:", round(best_roc_auc, 4))

k=3 получаем val ROC-AUC=0.7783
k=5 получаем val ROC-AUC=0.8613
k=7 получаем val ROC-AUC=0.8891
k=8 получаем val ROC-AUC=0.9312
k=9 получаем val ROC-AUC=0.9356
k=10 получаем val ROC-AUC=0.9371
k=11 получаем val ROC-AUC=0.9336
k=12 получаем val ROC-AUC=0.9298
k=13 получаем val ROC-AUC=0.9272
k=14 получаем val ROC-AUC=0.9328
k=15 получаем val ROC-AUC=0.9313

Лучший KNN
best k: 10
best val ROC-AUC: 0.9371


In [114]:
test_probs = best_knn.predict_proba(X_test_np)[:, 1]
test_pred = (test_probs >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_np, test_pred).ravel()

test_profit = tp * 5 - fp * 25 - fn * 5
test_roc_auc = roc_auc_score(y_test_np, test_probs)

print("k:", best_k)
print("ROC-AUC:", round(test_roc_auc, 4))
print("Profit:", test_profit)

k: 10
ROC-AUC: 0.8991
Profit: -130230


Получили обычным knn даже неплохое качество